# Lab 08: What Does Boosting Learn About Pokémon?

Use **XGBoost** and **CatBoost** to predict legendary status and primary type.
Investigate class imbalance, compare feature sets with and without potentially
revealing game mechanics, and examine which observations remain difficult as
trees are added. Connect these experiments to the AdaBoost and gradient-boosting
algorithms you have studied.

**Total:** 45 points. Run the supplied preparation cells, then complete the
remaining TODOs and written responses in the adjacent cells. Your interpretations
should follow your actual results.

Read the primer and its compact variable table, then follow each task's feature
instructions. You do not need prior Pokémon knowledge. `README.md` explains
installation and submission.


## Pokémon primer

Pokémon are fictional creatures that trainers collect and use in battles. In the main games, a Pokémon has a species, one or two elemental **types**, battle statistics, and other game characteristics. Types influence which attacks a Pokémon resists or is vulnerable to.

A trainer can gain experience, raise a Pokémon's level, and sometimes evolve it into a different species. The rows describe catalog entries, rather than individual trained Pokémon. Numerical battle columns are **base statistics**; an individual's level and training also affect its actual statistics. A blank second type means no second type is listed. **Legendary** is the source's designation; use the CSV label as given.

### Essential variable guide

| Group | Essential meaning and columns |
| --- | --- |
| Battle profile | hp, attack, defense, sp_attack, sp_defense, and speed describe base endurance, physical and special offense and defense, and move order. base_total is their exact sum in this snapshot, so the lab checks it and excludes it while retaining the six components. |
| Other numeric attributes | height_m, weight_kg, percentage_male, base_egg_steps, base_happiness, capture_rate, experience_growth, and generation describe size, species-level game mechanics, growth, or release generation. capture_rate is not a probability; one form-specific entry becomes missing when converted to numeric. |
| Types and matchups | type1 is the primary type; type2 is an optional secondary type. Each against_* column is an incoming damage multiplier: 1 is neutral, values above 1 indicate vulnerability, values between 0 and 1 indicate resistance, and 0 indicates immunity. |
| Labels and identifiers | is_legendary is the binary label for the legendary task. name is retained to inspect errors. Names, japanese_name, pokedex_number, classfication, and abilities are not model predictors in this lab. |

Physical and special moves use different offensive and defensive statistics. See the [official battle guide](https://diamondpearl.pokemon.com/en-gb/trainersguide/fundamentals/battling/). The [PokéAPI species documentation](https://pokeapi.co/docs/v2#pokemon-species) and [type relationships](https://pokeapi.co/docs/v2#types) provide optional background; this lab uses the supplied historical CSV without updating it.

## Evaluation protocol and supplied support

Use the local, unmodified snapshot described in [data/README.md](data/README.md).
lab_helpers.py supplies deterministic cleaning, class-aware splitting,
preprocessing for the baseline, and reporting helpers. You still create and fit
the models, choose thresholds, compare results, and explain your findings.

- **Training:** fit models and any learned preprocessing here. Derive class
  weights from these labels only.
- **Validation:** use for early stopping, threshold selection, feature-set and
  model choices, and all exploratory error/importance analysis.
- **Test:** evaluate only in Part 8, after recording your choices. Do not use
  test scores to select a library, threshold, feature set, or number of rounds.

The shared preprocessing removes primary-flying entries before either task.
Each task has its own fixed split, shared by every model for that task. Within
each class, approximately 60% of rows are used for training and 20% each for
validation and test. The primary-type task has 17 classes after preprocessing.
The splits evaluate new entries under this protocol, not necessarily new
evolutionary families, regions, or game generations.

Both boosters receive the same raw predictors, pandas categorical columns, and
numeric NaNs. XGBoost's enable_categorical=True enables its native categorical
support; CatBoost receives categorical column names. XGBoost also supports
categorical inputs—CatBoost is not the only library that does. The shallow tree
uses an imputation/one-hot pipeline fitted on training rows only.

The starter has no completed model outputs. TODO cells are intentionally
commented so setup can run before you complete the modeling. Replace the TODOs
with executable code; commented work does not satisfy a task.


In [ ]:
import time
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

from lab_helpers import (
    RANDOM_STATE, TYPE_NAMES, BATTLE_STATS, SHORTCUT_FEATURES,
    load_data, make_features, categorical_columns, make_splits, split_counts,
    baseline_preprocessor, binary_metrics, threshold_table, case_table,
    stage_probabilities, loss_curve,
)

pd.set_option("display.max_columns", 16)
plt.rcParams.update({"figure.figsize": (8, 5), "axes.grid": True})
EXCLUDED_PRIMARY_TYPE = "flying"
raw = load_data()
# Shared preprocessing removes this primary type from both modeling tasks.
raw = raw.loc[raw["type1"] != EXCLUDED_PRIMARY_TYPE].copy()
assert EXCLUDED_PRIMARY_TYPE not in set(raw["type1"])
display(pd.Series({name: version(name) for name in ["xgboost", "catboost", "scikit-learn", "pandas"]}, name="installed version"))


## Part 1: Understand the data and establish a baseline

### 1(a). Audit the preprocessed data

The supplied cell reports the row and column counts, legendary class counts and
percentages, majority-class accuracy, missing values, and primary-type counts.
Run the cell and inspect its output. What are the implications of these results?


In [ ]:
row_count, column_count = raw.shape
display(pd.Series({"rows": row_count, "columns": column_count}, name="dataset"))

legendary_counts = (
    raw["is_legendary"].astype(int).value_counts().reindex([0, 1], fill_value=0)
)
legendary_summary = pd.DataFrame({
    "count": legendary_counts,
    "percent": 100 * legendary_counts / row_count,
})
legendary_summary.index = legendary_summary.index.map(
    {0: "nonlegendary", 1: "legendary"}
)
display(legendary_summary)
display(pd.Series(
    {"majority_baseline_accuracy": legendary_counts.max() / row_count}
))

missing_counts = raw.isna().sum().rename("missing_count").to_frame()
display(missing_counts)

primary_type_counts = (
    raw["type1"].value_counts().rename_axis("primary_type").to_frame("count")
)
display(primary_type_counts)


**1(a): class balance — written response:**

Write your response here.


### 1(b). Inspect the cleaning decisions

The supplied cell identifies nonmissing `capture_rate` entries that cannot be
converted to a number and checks whether `base_total` equals the sum of
`BATTLE_STATS` for every row. Explain why we exclude the total while keeping its
components, and why a blank `type2` differs from a missing numeric measurement.


In [ ]:
capture_rate_numeric = pd.to_numeric(raw["capture_rate"], errors="coerce")
unparseable_capture = raw["capture_rate"].notna() & capture_rate_numeric.isna()
display(raw.loc[unparseable_capture, ["name", "capture_rate"]])

component_total = raw[list(BATTLE_STATS)].sum(axis=1)
total_matches = raw["base_total"].eq(component_total)
display(pd.Series({
    "rows_checked": len(raw),
    "all_base_totals_match": bool(total_matches.all()),
    "mismatch_count": int((~total_matches).sum()),
}))
if not total_matches.all():
    display(pd.DataFrame({
        "base_total": raw.loc[~total_matches, "base_total"],
        "component_sum": component_total.loc[~total_matches],
    }))


**1(b): cleaning and feature roles — written response:**

Write your response here.


### 1(c). Use the fixed legendary split

We use `make_features(raw, "legendary")` for the full predictor set: it retains
the battle statistics, other numeric attributes, `type1`, `type2`, and all
`against_*` matchup columns. The helper excludes identifiers, abilities,
descriptive text, `base_total`, and the target. For the restricted comparison,
`make_features(raw, "legendary", restricted=True)` additionally removes
`base_egg_steps`, `capture_rate`, and `base_happiness`. Both sets use the same
rows and labels; numeric NaNs are left for the boosters.

The supplied cell displays `split_counts(y_l, splits_l)`, partition sizes, and
checks for overlapping indices. Explain why class weights must be calculated
from training labels only, even though the split itself uses labels for
stratification.


In [ ]:
y_l = raw["is_legendary"].astype(int)
X_l = make_features(raw, "legendary")
X_r = make_features(raw, "legendary", restricted=True)
splits_l = make_splits(y_l)

X_train_l, y_train_l = X_l.loc[splits_l["train"]], y_l.loc[splits_l["train"]]
X_val_l, y_val_l = X_l.loc[splits_l["validation"]], y_l.loc[splits_l["validation"]]
X_test_l, y_test_l = X_l.loc[splits_l["test"]], y_l.loc[splits_l["test"]]
X_train_r, X_val_r, X_test_r = (
    X_r.loc[splits_l[part]] for part in ["train", "validation", "test"]
)

display(split_counts(y_l, splits_l))
partition_sizes = pd.Series(
    {name: len(indices) for name, indices in splits_l.items()},
    name="row_count",
)
display(partition_sizes.to_frame())

split_index_sets = {name: set(indices) for name, indices in splits_l.items()}
all_pairwise_disjoint = (
    split_index_sets["train"].isdisjoint(split_index_sets["validation"])
    and split_index_sets["train"].isdisjoint(split_index_sets["test"])
    and split_index_sets["validation"].isdisjoint(split_index_sets["test"])
)
all_rows_assigned = set().union(*split_index_sets.values()) == set(y_l.index)
display(pd.Series({
    "all_pairwise_disjoint": all_pairwise_disjoint,
    "all_rows_assigned": all_rows_assigned,
}))


**1(c): split and training-only weights — written response:**

Write your response here.


### 1(d). Fit a shallow-tree baseline

Create a `Pipeline` containing `baseline_preprocessor(X_train_l)` and
`DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE)`. Fit it on the
legendary training data. Use `predict_proba(X_val_l)[:, 1]` and `binary_metrics`
to report validation accuracy, average precision, precision, recall, and F1. Keep the fitted pipeline as `baseline_l`.

**Average precision (AP)** summarizes ranking performance across recall levels;
compute it from probability scores, not hard labels. In this lab, AP always
means scikit-learn's `average_precision_score`. Use
unweighted validation metrics even when training uses class weights.


In [ ]:
# baseline_l = Pipeline([
#     ("preprocess", baseline_preprocessor(X_train_l)),
#     ("tree", DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE)),
# ])
# TODO 1(d): Fit baseline_l and report binary_metrics on validation probabilities.


**1(d): baseline versus majority prediction — written response:**

Write your response here.


## Part 2: First XGBoost and CatBoost models

### 2(a). Fit unweighted models

Keep the settings fixed unless a later task explicitly changes them.

Create `xgb_plain` with `objective="binary:logistic"`, `eval_metric="logloss"`,
and `early_stopping_rounds=40`, using `XGB_SETTINGS`. Call `.fit` with training
features/labels and `eval_set=[(X_val_l, y_val_l)]`, `verbose=False`.

Create `cb_plain` with `loss_function="Logloss"`,
`eval_metric="Logloss"`, `early_stopping_rounds=40`,
`use_best_model=True`, and `cat_features=categorical_columns(X_train_l)`, using
`CB_SETTINGS`. Fit with `eval_set=(X_val_l, y_val_l)`.

Time each fit with `time.perf_counter()`. Report fit time and the selected number
of boosting rounds: `best_iteration + 1` for XGBoost and
`get_best_iteration() + 1` for CatBoost. Both iteration attributes are zero-based.
Early stopping monitors validation log loss; it does not use the final test set.


In [ ]:
XGB_SETTINGS = dict(
    n_estimators=400, max_depth=3, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
    tree_method="hist", device="cpu", enable_categorical=True,
    importance_type="gain", n_jobs=2, random_state=RANDOM_STATE,
)
CB_SETTINGS = dict(
    iterations=400, depth=3, learning_rate=0.05, l2_leaf_reg=3.0,
    task_type="CPU", thread_count=2, random_seed=RANDOM_STATE,
    allow_writing_files=False, verbose=False,
)

# TODO 2(a): Instantiate, time, and fit xgb_plain and cb_plain as specified.
# XGBClassifier(**XGB_SETTINGS, objective=..., eval_metric=..., early_stopping_rounds=40)
# CatBoostClassifier(**CB_SETTINGS, loss_function=..., eval_metric=...,
#                    cat_features=..., early_stopping_rounds=40, use_best_model=True)


### 2(b). Compare validation performance

Create a readable table of `binary_metrics` for the baseline and both boosters
at threshold 0.5. Save the fitted models and validation probabilities for later
comparisons. Describe differences in AP, minority-class recall, and precision.
Does a better accuracy necessarily mean a more useful legendary detector?


In [ ]:
# TODO 2(b): Predict validation probabilities and build the three-model metrics table.


**2(b): validation comparison — written response:**

Write your response here.


## Part 3: Change training weights and decision thresholds

### 3(a). Weight the minority class during training

Compute `positive_weight = n_nonlegendary_train / n_legendary_train` using
`y_train_l`. Fit two new models with the same settings and validation early
stopping as in Part 2:

- `xgb_weighted`: add `scale_pos_weight=positive_weight`.
- `cb_weighted`: add `class_weights=[1.0, positive_weight]`.

For weighted CatBoost, set `eval_metric="Logloss:use_weights=false"` so early
stopping monitors unweighted validation loss. The unweighted CatBoost model in
Part 2 uses `eval_metric="Logloss"`; its API rejects the `use_weights` option
when no training weights are present.

Use the same rows and features; do not combine these with resampling or other
automatic class-weight settings. Extend the validation comparison table with
the weighted models at threshold 0.5. Did weighting improve minority recall,
precision, and AP in this run? Report the result even if an expected gain fails
to occur. Weights affect fitting; weighted-model scores are not automatically
calibrated probabilities under the original class prevalence.


In [ ]:
# TODO 3(a): Calculate positive_weight from training labels only.
# TODO: Fit xgb_weighted and cb_weighted, then extend the validation table.


**3(a): effects of weighting — written response:**

Write your response here.


### 3(b). Choose thresholds using validation data

For each weighted model, call `threshold_table(y_val_l, probability)`. It evaluates
the fixed grid 0.05, 0.10, ..., 0.95. Choose the threshold with highest validation
F1; break ties by taking the **largest threshold**. Store your choices, for
example in `thresholds_full` with keys `"XGBoost"` and `"CatBoost"`.

For each model, report precision, recall, and F1 at 0.5 and at its chosen threshold.
Plot precision and recall against threshold for one model. Draw a labeled
validation confusion matrix at its chosen threshold. How do the false alarms
and missed legendary Pokémon change? Why does changing a threshold on these
same probabilities leave AP unchanged, whereas changing class weights can change AP?


In [ ]:
# TODO 3(b): Search the validation grid and store thresholds_full.
# Hint: sort_values(["f1", "threshold"], ascending=[False, False]).iloc[0]
# TODO: Compare default/chosen thresholds and make the requested plots.


**3(b): threshold trade-offs versus weighting — written response:**

Write your response here.


## Part 4: Remove candidate shortcuts

### 4. Which features made legendary prediction easy?

Refit **both weighted boosters** on `X_train_r`, which excludes
`base_egg_steps`, `capture_rate`, and `base_happiness`. Keep the same training
labels, class-weight ratio, other settings, and early-stopping rule; validation
features must now be `X_val_r`. Store `xgb_restricted` and `cb_restricted`.

Choose a new validation threshold for each restricted model using exactly the
same grid and tie rule; store `thresholds_restricted`. Make a four-row table
comparing both libraries' full and restricted weighted models. Include AP and
precision/recall/F1 at each model's validation-selected threshold.

Which performance measures changed most? What information was lost? Would these
variables be legitimate predictors if all catalog information were available,
and would the answer change if the goal were to infer legendary status from
physical characteristics and battle statistics? Explain why strong correlation
with the label alone does not establish data leakage.


In [ ]:
# TODO 4: Fit both restricted models, select their thresholds on validation,
# and compare full/restricted validation performance for each library.


**4: feature removal and the prediction setting — written response:**

Write your response here.


## Part 5: What happens as boosting adds trees?

### 5(a). One guided gradient calculation

For binary logistic loss, a boosted score $F(x)$ is a log-odds score, with
$p=1/(1+e^{-F})$. An observation's loss is

$$\ell(y,p)=-y\log p-(1-y)\log(1-p).$$

The negative gradient with respect to $F$ is $r=y-p$. Compute each observation's
loss and negative gradient in the supplied table. Which observation gives the
largest gradient magnitude? What direction should its score move? If positive
observations receive weight $w$, their weighted negative gradient is $w(y-p)$.
Recompute the negative gradients using the training-derived `positive_weight`.

Explain why the next tree supplies real-valued corrections rather than simply
refitting the original 0/1 labels. Distinguish this gradient-based mechanism from
AdaBoost's explicit update of observation weights. XGBoost also uses curvature
and regularization when constructing updates; this calculation isolates the
first-order intuition rather than reproducing its entire fitting algorithm.


In [ ]:
gradient_examples = pd.DataFrame({
    "case": ["confident correct nonlegendary", "uncertain legendary", "confident missed legendary"],
    "y": [0, 1, 1],
    "p_legendary": [0.05, 0.45, 0.05],
})
# TODO 5(a): Add individual log loss, negative gradient, and weighted negative gradient.


**5(a): which observation needs the largest correction? — written response:**

Write your response here.


### 5(b). Two controlled learning-rate experiments

Fit two **diagnostic XGBoost models** on the restricted training predictors with
learning rates **0.03 and 0.30**. Use the supplied diagnostic settings, binary
logistic objective, and `scale_pos_weight=positive_weight`. Run all **300 rounds**:
do not enable early stopping for these two diagnostic fits. This lets you see
behavior after validation performance may have stopped improving.

Use `loss_curve(model, X_train_r, y_train_l, X_val_r, y_val_l)` to obtain
unweighted training and validation log loss at checkpoints. Plot both curves
against rounds for each learning rate, with clearly labeled panels and a common
vertical scale. Report the checkpoint with lowest validation loss for each.

Which learning rate improves faster initially? Does either model show evidence
of overfitting? Why might a smaller learning rate need more rounds? Explain how
early stopping would help, using the curves; if the curves do not show clear
overfitting, say so. Only the learning rate differs between these two fits.


In [ ]:
DIAGNOSTIC_SETTINGS = XGB_SETTINGS | dict(
    n_estimators=300, max_depth=4, subsample=1.0, colsample_bytree=1.0,
)
# diagnostic_models = {}
# for rate in [0.03, 0.30]:
#     settings = DIAGNOSTIC_SETTINGS | {"learning_rate": rate}
#     TODO: Construct/fit the weighted binary XGBClassifier without early stopping.
#     TODO: Store the model and compute its loss_curve.
# TODO 5(b): Plot and compare the curves; report minimum-loss checkpoints.


**5(b): learning rate, rounds, and overfitting — written response:**

Write your response here.


### 5(c). Follow individual Pokémon through the ensemble

Use the **0.30 diagnostic model** to predict validation probabilities, then call
`case_table(raw, y_val_l, probability)` and show its five highest-loss rows.
The table preserves original row indices and includes names for discussion.
Individual log loss is `-log(p_true_class)`; it measures confidence in the true
label, independently of your chosen decision threshold and the training weights.

Choose three **distinct** validation observations: a low-loss example, a
highest-loss example, and an uncertain example with probability near 0.5.
Obtain checkpoint probabilities with `stage_probabilities(model, X_val_r)`.
For each selected row, plot **probability of its true class** against rounds:
use `p_legendary` when y=1 and `1 - p_legendary` when y=0. Label lines with names
and true labels, and keep the probability axis on [0, 1].

Does every row improve monotonically? Does a confidently wrong prediction
represent the same problem as an uncertain prediction? Explain why a reduction
in average training loss does not guarantee improvement for every validation
observation. These validation errors diagnose generalization; they are not the
residuals used to fit the next tree.


In [ ]:
# TODO 5(c): Build the highest-loss case table and select three distinct row indices.
# TODO: Get checkpoint probabilities and convert them to true-class probabilities.
# TODO: Plot the three trajectories and identify changes supported by the plot.


**5(c): uncertainty, confident errors, and individual trajectories — written response:**

Write your response here.


### 5(d). Do the libraries struggle with the same observations?

Use `case_table` for `xgb_restricted` and `cb_restricted` on their validation
probabilities. Compare the five highest-loss rows from each. Report how many row
indices overlap. Inspect the listed Pokémon's retained numerical attributes
using `raw.loc[...]` and describe one supported pattern or explain why there is
no clear pattern. Consider both unusually strong nonlegendary Pokémon and
legendary Pokémon with less typical attributes; do not assume these patterns occur.


In [ ]:
# TODO 5(d): Compare top-five loss sets and inspect retained attributes.


**5(d): shared and model-specific difficult cases — written response:**

Write your response here.


## Part 6: Predict primary type without matchups

### 6(a). Prepare the 17-class task

We use `make_features(raw, "type")` with `type1` as the target. The predictor
set retains the six battle statistics and other numeric attributes, including
categorical `generation`, while excluding `type1`, `type2`, `is_legendary`,
every `against_*` matchup column, and the common dropped identifier/text
fields. The supplied cell encodes the 17 observed primary types as class labels,
displays partition class counts by type name, and checks class coverage.

Explain why integer codes for the target represent nominal classes, not an
ordered numerical measurement.


In [ ]:
TYPE_NAMES_T = tuple(name for name in TYPE_NAMES if name in set(raw["type1"]))
assert len(TYPE_NAMES_T) == raw["type1"].nunique() == 17

type_encoder = LabelEncoder().fit(TYPE_NAMES_T)
y_t = pd.Series(
    type_encoder.transform(raw["type1"]),
    index=raw.index,
    name="primary_type_code",
)
X_t = make_features(raw, "type")
splits_t = make_splits(y_t)
X_train_t, y_train_t = X_t.loc[splits_t["train"]], y_t.loc[splits_t["train"]]
X_val_t, y_val_t = X_t.loc[splits_t["validation"]], y_t.loc[splits_t["validation"]]
X_test_t, y_test_t = X_t.loc[splits_t["test"]], y_t.loc[splits_t["test"]]

type_support = split_counts(y_t, splits_t)
type_support.index = type_encoder.inverse_transform(type_support.index.to_numpy())
display(type_support.rename_axis("primary_type"))
all_classes_present = bool(type_support.gt(0).all(axis=1).all())
display(pd.Series({"all_17_classes_in_each_partition": all_classes_present}))


**6(a): class coverage and label coding — written response:**

Write your response here.


### 6(b). Fit a baseline and both multiclass boosters

Fit baseline_t, a depth-two tree in the same kind of preprocessing pipeline
used in Part 1. Fit both boosters with the common base settings and the retained
17-class type training/validation rows:

- xgb_type: objective="multi:softprob", num_class=len(TYPE_NAMES_T),
  eval_metric="mlogloss", early_stopping_rounds=40.
- cb_type: loss_function="MultiClass", eval_metric="MultiClass",
  cat_features=categorical_columns(X_train_t), early_stopping_rounds=40,
  use_best_model=True.

Do not carry the binary positive-class weight or binary thresholds into this
multiclass experiment. Compare validation accuracy and macro-F1 for all three
models. Macro-F1 weights the 17 retained class-specific F1 scores equally,
rather than weighting them by how many observations each class contributes.


In [ ]:
# TODO 6(b): Fit baseline_t, xgb_type, and cb_type using training/validation only.
# TODO: Build the three-model accuracy and macro-F1 validation table.
# Use f1_score(..., average="macro", labels=np.arange(len(TYPE_NAMES_T)), zero_division=0).


**6(b): model comparison and macro-F1 — written response:**

Write your response here.


### 6(c). Which types are difficult?

Choose a boosting model by highest validation macro-F1; break a tie using higher
validation accuracy, then choose XGBoost if still tied. For that model, show a
named `classification_report` including class support and a row-normalized
validation confusion matrix. Use the encoder's class order consistently.
Identify two types with weak performance and two frequently confused type pairs,
reporting the relevant counts/support rather than interpreting colors alone.

Among its primary-type errors, how many predicted types equal the Pokémon's
recorded `type2`? Inspect `type2` only **after** predictions; it remains excluded
from features. Explain why such a prediction may be understandable but is still
incorrect for this lab's primary-type target. Does this suggest a limitation of
asking for only the primary type?


In [ ]:
# TODO 6(c): Report class-specific scores and a readable labeled confusion matrix.
# Hint: use figsize=(12, 10), normalize="true", and xticks_rotation=90.
# TODO: Compare primary-type mistakes with recorded secondary types for analysis only.


**6(c): confusion patterns, secondary types, and limited support — written response:**

Write your response here.


## Part 7: Which features guide the predictions?

### 7. Inspect built-in importance

For both libraries' **weighted full and restricted legendary models**, pair
`model.feature_importances_` with that model's input column names and plot the
top 10 as labeled horizontal bars. Use four panels, with each model's own scale
clearly labeled. List the top three features for each. Also plot the top 10 for
the boosting model selected for primary type in 6(c).

XGBoost is configured to report normalized **gain** importance. CatBoost reports
**PredictionValuesChange**, which summarizes changes in prediction values
associated with a feature's splits. These quantities have different definitions
and scales: compare rankings and substantive patterns, not bar heights across
libraries. Correlated or substitutable features can share importance, and these
are global summaries of fitted models rather than causal effects or explanations
of every individual prediction.

Which retained features replace the removed mechanics? Do the two prediction
tasks emphasize similar attributes? Relate one finding to the variable guide
and state one limitation of the importance analysis. No SHAP or permutation
importance is required.


In [ ]:
# Example pattern (adapt model and feature frame to each fit):
# importance = pd.Series(model.feature_importances_, index=X_train.columns)
# TODO 7: Make the four legendary panels and one type-model importance plot.
# TODO: Report top features without comparing incompatible importance magnitudes.


**7: important features, substitution, and interpretation limits — written response:**

Write your response here.


## Part 8: Lock choices and evaluate once

### 8(a). Record your choices before evaluating the test sets

Among the six main legendary boosters (unweighted full, weighted full, and
weighted restricted for each library), select highest **validation AP**. If tied,
prefer the restricted feature set, then XGBoost, then the unweighted fit. Exclude
the two fixed-length diagnostic models from this selection. Choose its threshold
on validation with the same grid/tie rule; compute it now if an unweighted model
is selected. For the type task, retain the model selected in 6(c).

Record the model names, full/restricted feature choice, validation selection
scores, legendary threshold, and selected boosting rounds. Do this **before**
any test predictions. Keep the fitted models as they are; do not refit on
training plus validation, which would change the probability model associated
with the chosen threshold. A small validation set and repeated comparisons can
make the selected validation score optimistic; the untouched test set provides
a separate check.


In [ ]:
# TODO 8(a): Record a compact frozen-choice table using validation results only.
# Keep the selected model objects, their feature sets, and the legendary threshold.


**8(a): frozen choices and their validation evidence — written response:**

Write your response here.


### 8(b). Final held-out evaluation and recommendation

Now evaluate the selected legendary model at its frozen threshold and the fitted
legendary shallow-tree baseline at 0.5. Use the appropriate full or restricted
test columns for each model. Report the same binary metrics and a labeled
confusion matrix for the selected booster. Evaluate the selected type model and
type baseline on their type test split using accuracy and macro-F1.

Write a 150–250 word recommendation supported by your results: how much the
boosters improved on the simple baselines, which task was harder, what the
feature-removal and difficult-observation analyses taught you, and what cannot
be concluded from this small historical snapshot. Include the legendary test
positive count and the least-supported type's test support (report ties). Do not
revise any choices after seeing test results, and do not claim that weighting or
boosting must win.


In [ ]:
# TODO 8(b): Evaluate frozen choices and baselines on the test sets once.
# TODO: Report binary/multiclass metrics and the selected legendary confusion matrix.


**8(b): final recommendation, 150–250 words — written response:**

Write your response here.


## Optional exploration ideas

Choose an idea only after completing the core lab:

- Can the six-stat **profile** predict more than the single `base_total`?
  Replace the components with their total rather than supplying both; make all
  experimental choices on validation data, not by revisiting test scores.
- Are the difficult Pokémon or importance rankings stable under several new
  training/validation splits inside the original development data?
- Does a decision stump versus a deeper base tree change what interactions the
  ensemble can represent? Propose or run a small controlled validation experiment.

## Submission check

- All TODOs have been replaced with executable code, and all written responses are complete.
- Tables identify validation versus test results; figures have readable titles,
  axes, legends, type names, and class labels as appropriate.
- Class weights, early stopping, threshold selection, and model selection use
  the specified partitions. Test results do not determine your choices.
- The supplied primary-type filter is applied, and the source CSV, helper, and
  environment files remain unchanged.
- Restart the kernel and run the completed notebook from top to bottom. Commit
  `lab-08.ipynb` and your generated `uv.lock`; do not commit `.venv` or model files.

## References

- Rounak Banik, [The Complete Pokémon Dataset](https://www.kaggle.com/datasets/rounakbanik/pokemon).
- [XGBoost categorical inputs](https://xgboost.readthedocs.io/en/release_3.1.0/tutorials/categorical.html),
  [scikit-learn interface and early stopping](https://xgboost.readthedocs.io/en/release_3.1.0/python/sklearn_estimator.html),
  and [boosted-tree objective](https://xgboost.readthedocs.io/en/release_3.1.0/tutorials/model.html).
- CatBoost [categorical features](https://catboost.ai/docs/en/features/categorical-features),
  [class weights and parameters](https://catboost.ai/docs/en/references/training-parameters/common),
  [staged probabilities](https://catboost.ai/docs/en/concepts/python-reference_catboostclassifier_staged_predict_proba),
  and [built-in importance](https://catboost.ai/docs/en/concepts/fstr).
- Scikit-learn [average precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html).
